In [1]:
import pandas as pd
import numpy as np
import random
import os
import time
import json 

# --- PARAMETER ALGORITMA MEMETIC ---
POP_SIZE = 100
GENERATIONS = 1000
MUTATION_RATE = 0.25
MAX_LOCAL_SEARCH_ITER = 150
BATAS_STAGNASI = 150

In [2]:
def hitung_jarak_rute(rute, matriks_jarak):
    """Menghitung total jarak dari urutan rute berdasarkan matriks jarak."""
    total = 0
    for i in range(len(rute) - 1):
        total += matriks_jarak[rute[i]][rute[i+1]]
    return total

def inisialisasi_populasi(pop_size, n):
    """Membuat populasi awal berisi rute acak yang diawali dan diakhiri di titik 0 (depot)."""
    populasi = []
    for _ in range(pop_size):
        rute_tengah = list(range(1, n))
        random.shuffle(rute_tengah)
        rute = [0] + rute_tengah + [0] # Kunci start dan end di titik 0
        populasi.append(rute)
    return populasi

def seleksi(populasi, matriks_jarak):
    """Memilih satu rute induk terbaik dari 3 rute acak (Tournament Selection)."""
    turnamen = random.sample(populasi, 3) 
    turnamen.sort(key=lambda x: hitung_jarak_rute(x, matriks_jarak))
    return turnamen[0]

def crossover(parent1, parent2):
    """Menggabungkan dua rute induk dengan Order Crossover (OX) agar tidak ada kota duplikat."""
    p1_tengah = parent1[1:-1]
    p2_tengah = parent2[1:-1]
    n_tengah = len(p1_tengah)
    
    if n_tengah < 2: return parent1[:]

    # Ambil potongan rute acak dari parent 1
    start, end = sorted(random.sample(range(n_tengah), 2))
    child_tengah = [-1] * n_tengah
    child_tengah[start:end] = p1_tengah[start:end]
    
    # Sisa slot kosong diisi menggunakan urutan dari parent 2
    p2_filtered = [x for x in p2_tengah if x not in child_tengah]
    idx = 0
    for i in range(n_tengah):
        if child_tengah[i] == -1:
            child_tengah[i] = p2_filtered[idx]
            idx += 1
            
    return [0] + child_tengah + [0]

def mutasi(rute, mutation_rate):
    """Menukar posisi dua kota secara acak di dalam rute (Swap Mutation)."""
    if random.random() < mutation_rate:
        rute_mutasi = rute.copy()
        idx1, idx2 = random.sample(range(1, len(rute)-1), 2)
        rute_mutasi[idx1], rute_mutasi[idx2] = rute_mutasi[idx2], rute_mutasi[idx1]
        return rute_mutasi
    return rute

def local_search_2opt(rute, max_iter, matriks_jarak):
    """Mengoptimalkan rute secara lokal dengan membalik sub-rute untuk mengurai persilangan jalan."""
    rute_terbaik = rute.copy()
    jarak_terbaik = hitung_jarak_rute(rute_terbaik, matriks_jarak)
    improved = True
    iterasi = 0
    
    while improved and iterasi < max_iter:
        improved = False
        for i in range(1, len(rute_terbaik) - 2):
            for j in range(i + 1, len(rute_terbaik) - 1):
                # Membalik urutan di antara titik i dan j
                rute_baru = rute_terbaik[:i] + rute_terbaik[i:j+1][::-1] + rute_terbaik[j+1:]
                jarak_baru = hitung_jarak_rute(rute_baru, matriks_jarak)
                
                if jarak_baru < jarak_terbaik:
                    rute_terbaik = rute_baru
                    jarak_terbaik = jarak_baru
                    improved = True
        iterasi += 1
    return rute_terbaik

In [3]:
def optimasi_per_wilayah(wilayah_list):
    """
    Menjalankan algoritma Memetic untuk setiap wilayah.
    Mengembalikan dictionary berisi rute terbaik per wilayah.
    """
    semua_rute_json = {}

    print("="*60)
    print("MEMULAI OPTIMASI MEMETIC ALGORITHM")
    print("="*60)

    for wilayah in wilayah_list:
        file_matriks = f'../data/matriks_jarak_{wilayah}.csv'
        
        # 1. Pengecekan File Data
        if not os.path.exists(file_matriks):
            print(f"Peringatan: File {file_matriks} tidak ditemukan. Skip wilayah {wilayah}.")
            continue
            
        # 2. Persiapan Data Matriks
        print(f"\n[{wilayah.upper()}] Membaca data...", end=" ")
        df_jarak = pd.read_csv(file_matriks, index_col=0)
        lokasi = list(df_jarak.columns)
        matriks_jarak = df_jarak.values
        jml_lokasi = len(lokasi)
        print(f"{jml_lokasi-1} Puskesmas siap.")

        # 3. Proses Evolusi Memetic
        start_time = time.time()
        populasi = inisialisasi_populasi(POP_SIZE, jml_lokasi)
        
        global_best_rute = None
        global_best_jarak = float('inf')
        stagnasi = 0
        jumlah_restart = 0

        print(f"[{wilayah.upper()}] Evolusi berjalan...", end=" ")
        
        for gen in range(GENERATIONS):
            populasi.sort(key=lambda x: hitung_jarak_rute(x, matriks_jarak))
            elite = populasi[0].copy()
            jarak_elite_sekarang = hitung_jarak_rute(elite, matriks_jarak)
            
            # Evaluasi stagnasi dan pengamanan elit
            if jarak_elite_sekarang < global_best_jarak:
                global_best_jarak = jarak_elite_sekarang
                global_best_rute = elite.copy()
                stagnasi = 0 
            else:
                stagnasi += 1
                
            # Reset populasi jika terjebak local optimum
            if stagnasi >= BATAS_STAGNASI:
                jumlah_restart += 1
                populasi = inisialisasi_populasi(POP_SIZE, jml_lokasi)
                populasi[0] = global_best_rute.copy() 
                stagnasi = 0
                continue 
                
            # Pembentukan generasi baru
            populasi_baru = [elite] 
            for _ in range(POP_SIZE - 1):
                p1 = seleksi(populasi, matriks_jarak)
                p2 = seleksi(populasi, matriks_jarak)
                child = crossover(p1, p2)
                child = mutasi(child, MUTATION_RATE)
                child_pintar = local_search_2opt(child, MAX_LOCAL_SEARCH_ITER, matriks_jarak)
                populasi_baru.append(child_pintar)

            populasi = populasi_baru
            
            # Indikator progres per 100 generasi
            if gen % 100 == 0:
                 print(f"{gen}.", end="", flush=True)

        end_time = time.time()
        processing_time = end_time - start_time

        # 4. Pencatatan Hasil Wilayah
        rute_nama = [lokasi[i] for i in global_best_rute]
        semua_rute_json[wilayah] = rute_nama
        
        print(f" Selesai!")
        print(f"[{wilayah.upper()}] Jarak Terpendek: {global_best_jarak:.3f} km")
        print(f"[{wilayah.upper()}] Waktu Komputasi: {processing_time:.2f} detik")

    return semua_rute_json


def simpan_ke_json(data_rute, output_path):
    """Menyimpan dictionary data rute ke dalam file berformat JSON."""
    print("\n"+"="*60)
    print("SEMUA WILAYAH SELESAI DIOPTIMASI.")

    # Otomatis membuat folder jika belum ada
    folder_output = os.path.dirname(output_path)
    os.makedirs(folder_output, exist_ok=True)

    with open(output_path, 'w') as json_file:
        json.dump(data_rute, json_file, indent=4) 

    print(f"File JSON berhasil disimpan di: {output_path}")
    print("="*60)

In [6]:
wilayah_list = ['barat', 'pusat', 'selatan', 'timur', 'utara']
path_simpan = '../output_json/rute_ma.json'

# 1. Jalankan algoritma untuk mendapatkan data rutenya
hasil_rute = optimasi_per_wilayah(wilayah_list)

MEMULAI OPTIMASI MEMETIC ALGORITHM

[BARAT] Membaca data... 12 Puskesmas siap.
[BARAT] Evolusi berjalan... 0.100.200.300.400.500.600.700.800.900. Selesai!
[BARAT] Jarak Terpendek: 44.983 km
[BARAT] Waktu Komputasi: 75.60 detik

[PUSAT] Membaca data... 8 Puskesmas siap.
[PUSAT] Evolusi berjalan... 0.100.200.300.400.500.600.700.800.900. Selesai!
[PUSAT] Jarak Terpendek: 26.111 km
[PUSAT] Waktu Komputasi: 36.34 detik

[SELATAN] Membaca data... 16 Puskesmas siap.
[SELATAN] Evolusi berjalan... 0.100.200.300.400.500.600.700.800.900. Selesai!
[SELATAN] Jarak Terpendek: 31.980 km
[SELATAN] Waktu Komputasi: 220.73 detik

[TIMUR] Membaca data... 14 Puskesmas siap.
[TIMUR] Evolusi berjalan... 0.100.200.300.400.500.600.700.800.900. Selesai!
[TIMUR] Jarak Terpendek: 28.243 km
[TIMUR] Waktu Komputasi: 144.31 detik

[UTARA] Membaca data... 13 Puskesmas siap.
[UTARA] Evolusi berjalan... 0.100.200.300.400.500.600.700.800.900. Selesai!
[UTARA] Jarak Terpendek: 34.851 km
[UTARA] Waktu Komputasi: 122.30 d

In [7]:
# Penyimpanan data ke file JSON

simpan_ke_json(hasil_rute, path_simpan)


SEMUA WILAYAH SELESAI DIOPTIMASI.
File JSON berhasil disimpan di: ../output_json/rute_ma.json
